In [1]:
import pyarrow.dataset as ds

# Load first 1,000 rows efficiently
dataset = ds.dataset("cohere_wikipedia_en_1M.parquet", format="parquet")
df = next(dataset.to_batches(batch_size=1000)).to_pandas()

# Inspect dataset structure
print("DataFrame Shape:", df.shape)
print("\nColumns:", df.columns.tolist())


DataFrame Shape: (1000, 5)

Columns: ['_id', 'url', 'title', 'text', 'emb']


In [2]:
df.head(10)

,_id,url,title,text,emb
0,20231101.en_13194570_0,https://en.wikipedia.org/wiki/British%20Arab%2...,British Arab Commercial Bank,The British Arab Commercial Bank PLC (BACB) is...,"[0.0017852783203125, -0.040496826171875, 0.004..."
1,20231101.en_13194570_1,https://en.wikipedia.org/wiki/British%20Arab%2...,British Arab Commercial Bank,"BACB has a head office in London, and three re...","[0.00518798828125, -0.02630615234375, -0.01149..."
2,20231101.en_13194570_2,https://en.wikipedia.org/wiki/British%20Arab%2...,British Arab Commercial Bank,"The bank provides services of trade finance, t...","[0.00818634033203125, -0.006195068359375, 0.00..."
3,20231101.en_13194570_3,https://en.wikipedia.org/wiki/British%20Arab%2...,British Arab Commercial Bank,The bank was founded in 1972 as UBAF Limited a...,"[0.0135650634765625, -0.046417236328125, -0.01..."
4,20231101.en_13194570_4,https://en.wikipedia.org/wiki/British%20Arab%2...,British Arab Commercial Bank,"In 2009, Commercial Bank of Egypt sold its 8% ...","[0.0007767677307128906, -0.030059814453125, -0..."
5,20231101.en_13194570_5,https://en.wikipedia.org/wiki/British%20Arab%2...,British Arab Commercial Bank,"In 2010, HSBC sold its 49% shareholding to the...","[-0.02874755859375, -0.0421142578125, -0.02142..."
6,20231101.en_13194570_6,https://en.wikipedia.org/wiki/British%20Arab%2...,British Arab Commercial Bank,"In 2016, the bank opened a new representative ...","[0.02783203125, -0.023834228515625, -0.0167846..."
7,20231101.en_13194570_7,https://en.wikipedia.org/wiki/British%20Arab%2...,British Arab Commercial Bank,BACB aims to facilitate cross-border trade thr...,"[0.01129150390625, -0.0017490386962890625, -0...."
8,20231101.en_13194570_8,https://en.wikipedia.org/wiki/British%20Arab%2...,British Arab Commercial Bank,"Most notably, the bank works with Saf Cacao in...","[0.00457763671875, -0.01007843017578125, -0.02..."
9,20231101.en_13194570_9,https://en.wikipedia.org/wiki/British%20Arab%2...,British Arab Commercial Bank,While the majority of the bank's clients are A...,"[0.01453399658203125, -0.0289154052734375, 0.0..."


In [4]:
df.iloc[1]['text']

"BACB has a head office in London, and three representative offices in Algiers in Algeria, Tripoli in Libya and Abidjan in the Cote D'Ivoire. The bank has 17 sister banks across Europe, Asia and Africa. It is owned by three main shareholders - the Libyan Foreign Bank (87.80%), Banque Centrale Populaire (6.10%) and Banque Extérieure d'Algérie (6.10%)."

#Inserting rows to Postgres DB

In [1]:
import time
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from pgvector.psycopg2 import register_vector
import pyarrow.dataset as ds

# 1. Database Connection Setup
DB_CONFIG = {
    "dbname": "vector_search",
    "user": "postgres",
    "password": "postgres",
    "host": "localhost",
    "port": 5432,
}

PARQUET_PATH = "cohere_wikipedia_en_1M.parquet"
BATCH_SIZE = 5000

conn = psycopg2.connect(**DB_CONFIG)

# Register pgvector extension with psycopg2
register_vector(conn)
cursor = conn.cursor()

# 2. Batch Ingestion Loop
parquet_dataset = ds.dataset(PARQUET_PATH, format="parquet")

insert_query = """
    INSERT INTO wikipedia_embeddings (wiki_id, url, title, text, emb)
    VALUES %s
"""

start_time = time.time()
total_inserted = 0

print("Starting bulk insertion into PostgreSQL...")

for batch in parquet_dataset.to_batches(batch_size=BATCH_SIZE):
    df_batch = batch.to_pandas()

    records = []
    for _, row in df_batch.iterrows():
        emb = row.get("emb")
        
        # Skip or handle rows with missing/empty vectors
        if emb is None or len(emb) == 0:
            continue

        # Format row tuple (pgvector adapter handles list -> vector natively)
        wiki_id = int(row["id"]) if "id" in row and pd.notna(row["id"]) else None
        url = str(row["url"]) if pd.notna(row.get("url")) else ""
        title = str(row["title"]) if pd.notna(row.get("title")) else ""
        text = str(row["text"]) if pd.notna(row.get("text")) else ""
        
        records.append((wiki_id, url, title, text, [float(x) for x in emb]))

    if records:
        execute_values(cursor, insert_query, records, page_size=BATCH_SIZE)
        conn.commit()
        total_inserted += len(records)

    elapsed = time.time() - start_time
    print(f"Inserted {total_inserted:,} rows ({elapsed:.1f}s)")

cursor.close()
conn.close()

print(f"\nSuccessfully loaded {total_inserted:,} rows into 'wikipedia_embeddings'!")

Starting bulk insertion into PostgreSQL...
Inserted 5,000 rows (10.2s)
Inserted 10,000 rows (19.3s)
Inserted 15,000 rows (27.6s)
Inserted 20,000 rows (36.2s)
Inserted 25,000 rows (44.6s)
Inserted 30,000 rows (52.9s)
Inserted 35,000 rows (61.3s)
Inserted 40,000 rows (69.5s)
Inserted 45,000 rows (77.9s)
Inserted 50,000 rows (86.2s)
Inserted 55,000 rows (94.5s)
Inserted 60,000 rows (102.8s)
Inserted 65,000 rows (111.2s)
Inserted 70,000 rows (119.5s)
Inserted 75,000 rows (127.9s)
Inserted 80,000 rows (136.3s)
Inserted 85,000 rows (144.8s)
Inserted 90,000 rows (153.1s)
Inserted 95,000 rows (161.5s)
Inserted 100,000 rows (170.5s)
Inserted 105,000 rows (179.4s)
Inserted 110,000 rows (187.8s)
Inserted 115,000 rows (196.4s)
Inserted 120,000 rows (204.9s)
Inserted 125,000 rows (213.6s)
Inserted 130,000 rows (222.2s)
Inserted 135,000 rows (230.9s)
Inserted 140,000 rows (239.3s)
Inserted 145,000 rows (247.6s)
Inserted 150,000 rows (256.1s)
Inserted 155,000 rows (264.4s)
Inserted 160,000 rows (272.